<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio propuesto, Modulo 3 Tema 2 - Kafka + Spark Structured Streaming</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>

## Kafka + Spark Structured Streaming

El objetivo es modernizar el ejemplo de este repositorio `https://github.com/dbusteed/kafka-spark-streaming-example` usando Docker Compose, Kafka en modo KRaft, Spark Structured Streaming, eventos JSON y salida en Parquet.

### Objetivos

1. Generar eventos simulados tipo tweet en formato JSON desde un productor Python.
2. Publicar los eventos en un topic Kafka llamado `tweets`.
3. Consumir el topic desde Spark Structured Streaming.
4. Validar y transformar los eventos recibidos.
5. Guardar eventos validos en Parquet.
6. Enviar eventos invalidos a una zona de cuarentena.
7. Generar metricas por microbatch para comprobar el procesamiento.


## 1. Ficheros disponibles

La carpeta del laboratorio contiene estos ficheros principales:

- `Dockerfile`: imagen del contenedor de laboratorio con Python, Java y PySpark.
- `docker-compose.yml`: servicios `kafka` y `lab`.
- `crear_laboratorio.sh`: script de apoyo para recrear la estructura si faltan `app/`, `output/` o `checkpoint/`.
- `KAFKA_SPARK_STREAMING_EXAMPLE_ACTUALIZADO.md`: guia ampliada del ejercicio.

Si tu copia local ya contiene `app/`, `output/` y `checkpoint/`, no hace falta ejecutar el script de creacion. Si falta la aplicacion Python, usa `crear_laboratorio.sh` como mecanismo de recuperacion.


In [3]:
from pathlib import Path

base_dir = Path.cwd()
print("Directorio actual:", base_dir)

for name in ["Dockerfile", "docker-compose.yml", "crear_laboratorio.sh", "app", "output", "checkpoint"]:
    path = base_dir / name
    print(f"{name:24} ->", "OK" if path.exists() else "NO EXISTE")


Directorio actual: /Users/noelia/PycharmProjects/PFG/modulo3/Tema2
Dockerfile               -> OK
docker-compose.yml       -> OK
crear_laboratorio.sh     -> OK
app                      -> NO EXISTE
output                   -> NO EXISTE
checkpoint               -> NO EXISTE


## 2. Requisitos

Necesitas Docker y Docker Compose. En Ubuntu/Debian puedes comprobarlo con los comandos siguientes. En Windows o macOS, usa Docker Desktop y ejecuta los comandos desde una terminal con acceso a Docker.


In [4]:
!docker --version
!docker compose version
!docker run --rm hello-world


Docker version 29.5.3, build d1c06ef
Docker Compose version v5.1.4
Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world

e6a49ef1: Pulling fs layer 
Digest: sha256:96498ffd522e70807ab6384a5c0485a79b9c7c08ca79ba08623edcad1054e62d
Status: Downloaded newer image for hello-world:latest

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and

## 3. Crear la estructura del laboratorio

Ejecuta este paso solo si faltan carpetas o ficheros de la aplicacion, por ejemplo `app/producer_json.py`, `app/streaming_job.py` o `app/validate_outputs.py`.

El script crea:

- `app/`
- `output/`
- `checkpoint/`
- `requirements.txt`
- scripts Python del productor, consumidor y validador


In [5]:
# Ejecutar solo si necesitas recrear la aplicacion del laboratorio.
# En Windows puede requerir Git Bash, WSL o una terminal compatible con bash.
!bash crear_laboratorio.sh


Laboratorio creado correctamente.
Ficheros:
./Dockerfile
./KAFKA_SPARK_STREAMING_EXAMPLE_ACTUALIZADO.md
./M3T2.ipynb
./M3T2L1.html
./M3T2L2.html
./M3T2L3.html
./M3T2L4.html
./M3T2L5.html
./M3T2auto.html
./app/producer_json.py
./app/streaming_job.py
./app/validate_outputs.py
./crear_laboratorio.sh
./docker-compose.yml
./index.html
./requirements.txt


## 4. Construir la imagen del laboratorio

La primera ejecucion puede tardar varios minutos porque descarga Python, Java, PySpark y dependencias.

Resultado esperado: construccion sin errores y mensaje final similar a `DONE` o `Successfully built`.


In [8]:
!docker compose build


[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.1s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 530B                                             0.0s
[+] Building 0.3s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 530B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 459B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim-bookw  0.2s
[+] Building 0.4s (2/3)                                                         
 => [internal] load local bake defin

## 5. Arrancar Kafka

Arranca solo el servicio Kafka y espera a que el contenedor aparezca como `healthy`.


In [7]:
!docker compose up -d kafka
!docker compose ps


[+] up 1/1
 ✔ Container m3t2-kafka Running                                             0.0s

What's next:
    Filter, search, and stream logs from all your Compose services
    in one place with Docker Desktop's Logs view. ]8;;docker-desktop://dashboard/logs?appId=tema2\docker-desktop://dashboard/logs?appId=tema2]8;;\
NAME         IMAGE                COMMAND                  SERVICE   CREATED         STATUS                   PORTS
m3t2-kafka   apache/kafka:4.2.0   "/__cacert_entrypoin…"   kafka     5 minutes ago   Up 5 minutes (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp


## 6. Crear el topic `tweets`

El topic se crea con tres particiones y factor de replicacion 1, adecuado para un laboratorio local con un unico broker.


In [8]:
!docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic tweets --partitions 3 --replication-factor 1


Created topic tweets.


Verifica que el topic existe y que tiene tres particiones.


In [9]:
!docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --describe --topic tweets


Topic: tweets	TopicId: a_W6Wn4oQpWFF-xILSan4w	PartitionCount: 3	ReplicationFactor: 1	Configs: min.insync.replicas=1
	Topic: tweets	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: tweets	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: tweets	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


## 7. Producir eventos JSON

El productor genera eventos simulados tipo tweet y los publica en el topic `tweets`.

Salida esperada:

```text
Eventos enviados: 25
Eventos enviados: 50
...
Finalizado. Total eventos enviados: 300
```


In [10]:
!docker compose run --rm lab python /workspace/app/producer_json.py --bootstrap-server kafka:9092 --topic tweets --messages 300 --sleep 0.01


[+] create 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+] start 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+]  1/1
 ✔ Container m3t2-kafka Running                                             0.0s
Container m3t2-kafka Waiting 
Container m3t2-kafka Healthy 
Container tema2-lab-run-9e16fb0c1fdd Creating 
Container tema2-lab-run-9e16fb0c1fdd Created 
Eventos enviados: 25
Eventos enviados: 50
Eventos enviados: 75
Eventos enviados: 100
Eventos enviados: 125
Eventos enviados: 150
Eventos enviados: 175
Eventos enviados: 200
Eventos enviados: 225
Eventos enviados: 250
Eventos enviados: 275
Eventos enviados: 300
Finalizado. Total eventos enviados: 300


## 8. Ver mensajes en Kafka

Consume algunos mensajes desde el principio del topic para comprobar que se han publicado correctamente. Deben verse mensajes JSON con campos como `event_id`, `event_time`, `source`, `text`, `hashtags` y `mentions`.


In [11]:
!docker compose exec kafka /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic tweets --from-beginning --max-messages 5


{"event_id": "abf55181-a76d-49ce-95d1-9a364bb23ff9", "event_time": "2026-06-22T15:43:11Z", "source": "m3t2-json-producer", "text": "window kafka parquet parquet data window streaming analytics streaming @spark_user", "hashtags": [], "mentions": ["@spark_user"]}
{"event_id": "b39d4d07-67f1-46b8-8207-4c4d53bb422a", "event_time": "2026-06-22T15:43:11Z", "source": "m3t2-json-producer", "text": "parquet data analytics pipeline spark data python data streaming python topic checkpoint kafka event #spark @data_team", "hashtags": ["#spark"], "mentions": ["@data_team"]}
{"event_id": "ec58f036-441d-47ed-8d7b-830424854468", "event_time": "2026-06-22T15:43:11Z", "source": "m3t2-json-producer", "text": "streaming pipeline latency kafka parquet pipeline event analytics latency producer event event #kafka @kafka_lab", "hashtags": ["#kafka"], "mentions": ["@kafka_lab"]}
{"event_id": "b8ef4cb9-d673-477a-a7c1-be2eb0baf31e", "event_time": "2026-06-22T15:43:11Z", "source": "m3t2-json-producer", "text": "sp

## 9. Ejecutar Spark Structured Streaming

Spark consume el topic, valida los eventos, separa validos e invalidos, calcula metricas y escribe resultados en Parquet.

La primera ejecucion puede tardar porque Spark descarga el conector Kafka desde Maven.

Salida esperada:

```text
Batch 0: validos=300, invalidos=0, metricas=1
Consulta finalizada.
```


In [12]:
!docker compose run --rm lab spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 /workspace/app/streaming_job.py --bootstrap-server kafka:9092 --topic tweets --output /workspace/output --checkpoint /workspace/checkpoint/tweets_stream


[+] create 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+] start 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+]  1/1
 ✔ Container m3t2-kafka Running                                             0.0s
Container m3t2-kafka Waiting 
Container m3t2-kafka Healthy 
Container tema2-lab-run-4f8941ff6174 Creating 
Container tema2-lab-run-4f8941ff6174 Created 
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-299c57ed-a93c-4f27-afcb-550db7626c6f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12

## 10. Validar resultados

El validador lee las salidas Parquet generadas por Spark y muestra conteos y muestras de datos.

Resultado esperado:

```text
== tweets_valid ==
Registros: 300

== tweets_quarantine ==
Registros: 0

== tweets_metrics ==
Registros: 1
```


In [13]:
!docker compose run --rm lab python /workspace/app/validate_outputs.py


[+] create 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+] start 1/1
 ✔ Container m3t2-kafka Running                                             0.0s
[+]  1/1
 ✔ Container m3t2-kafka Running                                             0.0s
Container m3t2-kafka Waiting 
Container m3t2-kafka Healthy 
Container tema2-lab-run-23834de58f72 Creating 
Container tema2-lab-run-23834de58f72 Created 
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 15:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

== tweets_valid ==
Registros: 300
+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Tambien puedes revisar los ficheros generados desde la maquina anfitriona.


In [14]:
from pathlib import Path

for folder in [Path("output"), Path("checkpoint")]:
    print(f"\n== {folder} ==")
    if not folder.exists():
        print("No existe")
        continue
    files = sorted(p for p in folder.rglob("*") if p.is_file())
    if not files:
        print("Sin ficheros")
        continue
    for file in files:
        print(file)



== output ==
output/tweets_metrics/._SUCCESS.crc
output/tweets_metrics/.part-00000-821f7b94-fa10-4096-a79c-adcb3ff9d327-c000.snappy.parquet.crc
output/tweets_metrics/_SUCCESS
output/tweets_metrics/part-00000-821f7b94-fa10-4096-a79c-adcb3ff9d327-c000.snappy.parquet
output/tweets_quarantine/._SUCCESS.crc
output/tweets_quarantine/.part-00000-c7054bf6-bef1-40e4-9e9a-478cceafc21f-c000.snappy.parquet.crc
output/tweets_quarantine/_SUCCESS
output/tweets_quarantine/part-00000-c7054bf6-bef1-40e4-9e9a-478cceafc21f-c000.snappy.parquet
output/tweets_valid/._SUCCESS.crc
output/tweets_valid/.part-00000-5b6e287a-7328-481a-b7c0-77c0961c4960-c000.snappy.parquet.crc
output/tweets_valid/.part-00001-5b6e287a-7328-481a-b7c0-77c0961c4960-c000.snappy.parquet.crc
output/tweets_valid/.part-00002-5b6e287a-7328-481a-b7c0-77c0961c4960-c000.snappy.parquet.crc
output/tweets_valid/.part-00003-5b6e287a-7328-481a-b7c0-77c0961c4960-c000.snappy.parquet.crc
output/tweets_valid/_SUCCESS
output/tweets_valid/part-00000-5b6e

## 11. Repetir desde cero

Para limpiar resultados y checkpoints de una ejecucion anterior, borra `output/` y `checkpoint/` antes de producir eventos y ejecutar de nuevo el job Spark.


In [16]:
# Ejecutar solo si quieres reiniciar completamente la prueba.
# En Linux/macOS/WSL:
!rm -rf output/* checkpoint/*


## 12. Parar el laboratorio

Para parar los contenedores:


In [15]:
!docker compose down


[+] down 0/1
 ⠋ Container m3t2-kafka Stopping                                            0.1s
[+] down 0/1
 ⠙ Container m3t2-kafka Stopping                                            0.2s
[+] down 0/1
 ⠹ Container m3t2-kafka Stopping                                            0.3s
[+] down 0/1
 ⠸ Container m3t2-kafka Stopping                                            0.4s
[+] down 0/1
 ⠼ Container m3t2-kafka Stopping                                            0.5s
[+] down 0/1
 ⠴ Container m3t2-kafka Stopping                                            0.6s
[+] down 0/1
 ⠦ Container m3t2-kafka Stopping                                            0.7s
[+] down 0/1
 ⠧ Container m3t2-kafka Stopping                                            0.8s
[+] down 0/1
 ⠇ Container m3t2-kafka Stopping                                            0.9s
[+] down 1/2
 ✔ Container m3t2-kafka  Removed                                            0.9s
 ⠋ Network tema2_default Removing                           

Para borrar tambien los volumenes internos de Kafka:


In [ ]:
# Ejecutar solo si quieres borrar tambien los datos internos de Kafka.
# !docker compose down -v


## 13. Ejercicios de mejora con soluciones

1. **Latencia extremo a extremo:** calcula la diferencia entre `event_ts` y `processed_time`. Guarda media, minimo y maximo por microbatch.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 1</strong></summary>

En `streaming_job.py`, dentro de `process_batch`, anade una columna de latencia al DataFrame `valid`:

```python
from pyspark.sql.functions import unix_timestamp

valid = valid.withColumn(
    "latency_seconds",
    unix_timestamp("processed_time") - unix_timestamp("event_ts")
)
```

Despues incorpora las metricas en el agregado por microbatch:

```python
metrics = (
    valid
    .groupBy("source")
    .agg(
        count("*").alias("events"),
        avg("length").alias("avg_length"),
        avg("words").alias("avg_words"),
        avg("latency_seconds").alias("avg_latency_seconds"),
        min("latency_seconds").alias("min_latency_seconds"),
        max("latency_seconds").alias("max_latency_seconds")
    )
    .withColumn("batch_id", lit(batch_id))
)
```

La latencia mide el retraso aproximado entre el tiempo del evento y el momento en que Spark lo procesa.

</details>

2. **Particionado:** compara una ejecucion con un topic de una particion y otra con tres particiones. Analiza tiempo total, distribucion de offsets, numero de ficheros generados y efecto sobre el paralelismo.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 2</strong></summary>

Crea dos topics distintos para no mezclar resultados:

```bash
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh   --bootstrap-server kafka:9092   --create --if-not-exists   --topic tweets_1p   --partitions 1   --replication-factor 1

docker compose exec kafka /opt/kafka/bin/kafka-topics.sh   --bootstrap-server kafka:9092   --create --if-not-exists   --topic tweets_3p   --partitions 3   --replication-factor 1
```

Ejecuta el productor y el job Spark cambiando `--topic`. Usa carpetas de salida y checkpoint diferentes para cada prueba:

```bash
--topic tweets_1p --output /workspace/output_1p --checkpoint /workspace/checkpoint/tweets_1p
--topic tweets_3p --output /workspace/output_3p --checkpoint /workspace/checkpoint/tweets_3p
```

Compara:

```bash
find output_1p -type f | wc -l
find output_3p -type f | wc -l
```

Con mas particiones Kafka puede haber mas paralelismo de lectura, pero en un laboratorio pequeno tambien puede aumentar el numero de tareas y ficheros pequenos.

</details>

3. **Eventos invalidos:** modifica el productor para generar un 5% de mensajes sin `event_id`, sin `text` o con JSON mal formado. Comprueba que aparecen en `output/tweets_quarantine`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 3</strong></summary>

En `producer_json.py`, antes de enviar el evento, introduce una probabilidad de corrupcion:

```python
if random.random() < 0.05:
    error_type = random.choice(["missing_event_id", "missing_text", "malformed_json"])
    if error_type == "missing_event_id":
        event.pop("event_id", None)
        producer.send(args.topic, key=str(uuid.uuid4()), value=event)
    elif error_type == "missing_text":
        event.pop("text", None)
        producer.send(args.topic, key=event.get("event_id", str(uuid.uuid4())), value=event)
    else:
        producer.send(args.topic, key=str(uuid.uuid4()), value="{json_mal_formado")
else:
    producer.send(args.topic, key=event["event_id"], value=event)
```

Para mensajes mal formados puede ser necesario adaptar el serializador, porque el productor actual serializa siempre con `json.dumps`. Una alternativa didactica es enviar manualmente bytes para ese caso.

Despues ejecuta el validador y comprueba `tweets_quarantine`.

</details>

4. **Salida agregada por ventana:** genera metricas por ventana de evento de un minuto con total de eventos, longitud media, hashtags y menciones.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 4</strong></summary>

Puedes crear un agregado por ventana antes del `foreachBatch` o dentro del batch si trabajas sobre datos ya parseados. En streaming estructurado:

```python
window_metrics = (
    parsed
    .withWatermark("event_ts", "10 minutes")
    .groupBy(window("event_ts", "1 minute"), "source")
    .agg(
        count("*").alias("events"),
        avg(length("text")).alias("avg_length"),
        spark_sum(size("hashtags")).alias("hashtags"),
        spark_sum(size("mentions")).alias("mentions")
    )
)
```

Recuerda importar `window`:

```python
from pyspark.sql.functions import window
```

Luego escribe `window_metrics` en Parquet con su propio checkpoint.

</details>

5. **Migracion conceptual:** documenta que partes del ejemplo original se han sustituido: ZooKeeper por KRaft, Spark Streaming clasico por Structured Streaming, Hive textfile por Parquet, texto plano por JSON y ejecucion manual por Docker Compose.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 5</strong></summary>

Una respuesta correcta puede resumirse asi:

| Componente original | Sustitucion actual | Motivo |
|---|---|---|
| ZooKeeper | Kafka KRaft | Kafka moderno puede gestionar metadatos sin ZooKeeper. |
| Spark Streaming clasico | Spark Structured Streaming | API recomendada, basada en DataFrames y SQL. |
| Kafka 0.8 connector | `spark-sql-kafka-0-10` | Conector moderno para Structured Streaming. |
| Hive textfile | Parquet | Formato columnar, tipado y eficiente para analitica. |
| Texto plano | JSON con esquema | Eventos semiestructurados, validables y extensibles. |
| Instalacion manual | Docker Compose | Reproducibilidad y menor friccion de instalacion. |

La idea didactica se mantiene: productor, broker Kafka, consumidor Spark y almacenamiento analitico.

</details>

6. **Sink alternativo:** escribe los eventos enriquecidos en otro topic Kafka llamado `tweets_enriched`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 6</strong></summary>

Primero crea el topic:

```bash
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh   --bootstrap-server kafka:9092   --create --if-not-exists   --topic tweets_enriched   --partitions 3   --replication-factor 1
```

Prepara el DataFrame de salida con `key` y `value`:

```python
from pyspark.sql.functions import to_json, struct

out = (
    valid
    .selectExpr("CAST(event_id AS STRING) AS key", "*")
    .withColumn("value", to_json(struct("*")))
    .select("key", "value")
)
```

Escribe en Kafka:

```python
query = (
    out.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", args.bootstrap_server)
    .option("topic", "tweets_enriched")
    .option("checkpointLocation", "/workspace/checkpoint/tweets_enriched")
    .start()
)
```

En el script actual, como se usa `foreachBatch`, tambien puedes escribir a Kafka dentro de `process_batch` usando `valid.write.format("kafka")` en modo batch.

</details>

7. **Idempotencia:** ejecuta dos veces el job con el mismo checkpoint y comprueba que no reprocesa los mismos offsets. Despues borra `checkpoint/`, ejecuta otra vez y explica que ocurre.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 7</strong></summary>

Con el mismo checkpoint, Spark conserva los offsets ya procesados. Si ejecutas de nuevo el job sin producir eventos nuevos, no deberia reprocesar los mismos mensajes.

Prueba:

```bash
docker compose run --rm lab spark-submit ... --checkpoint /workspace/checkpoint/tweets_stream
```

Ejecuta el mismo comando una segunda vez y revisa que no se dupliquen resultados si no hay nuevos offsets disponibles.

Despues borra el checkpoint:

```bash
rm -rf checkpoint/tweets_stream
```

Al ejecutar de nuevo con `startingOffsets=earliest`, Spark pierde la memoria de offsets procesados y vuelve a leer desde el principio del topic. Por eso pueden duplicarse registros en la salida si no se limpia tambien `output/` o si no hay una estrategia de deduplicacion/idempotencia en el sink.

</details>


## 14. Problemas frecuentes

### Kafka no esta listo

Espera a que el healthcheck termine y revisa el estado:

```bash
docker compose ps
docker compose logs kafka --tail=50
```

### `UnknownHostException: kafka`

Este error suele aparecer cuando ejecutas Spark fuera de Docker. La guia esta pensada para ejecutar productor y Spark dentro del servicio `lab`, usando `kafka:9092` como bootstrap server.

### Error descargando paquetes Maven

El primer `spark-submit --packages` necesita internet para descargar el conector Spark-Kafka. Repite cuando haya red o configura un repositorio Maven interno.

### Error instalando `openjdk-17-jre-headless`

Si `docker compose build` falla con `exit code: 100`, aseg?rate de que el `Dockerfile` use:

```dockerfile
FROM python:3.11-slim-bookworm
```

Despues ejecuta:

```bash
docker compose build --no-cache lab
```

### No aparecen nuevos datos

Si usas el mismo checkpoint, Spark continua desde el ultimo offset procesado. Para una prueba desde cero:

```bash
rm -rf checkpoint/* output/*
```

### Hay muchos ficheros pequenos

Es normal en laboratorios con microbatches. En produccion se ajustan particiones, frecuencia de trigger y procesos de compactacion.


## 15. Resultado esperado

Tras ejecutar los pasos principales, `validate_outputs.py` debe mostrar:

- registros validos en `tweets_valid`;
- cero o mas registros invalidos en `tweets_quarantine`;
- metricas por `source` en `tweets_metrics`.

Ejemplo:

```text
== tweets_valid ==
Registros: 300

== tweets_quarantine ==
Registros: 0

== tweets_metrics ==
Registros: 1
```
